# Lab 5 — Linear, Neighbor, and Probabilistic Classification
**Coverage:** Chapters 10–11

This notebook is one of the ten course labs. Complete the core activities in order; transfer activities are optional extensions inside the same lab and do not create additional lab numbers.


## Part A — Logistic regression and threshold policies on Titanic
**Core activity.**


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "titanic_dataset" / "Titanic-Dataset.csv"
df = pd.read_csv(train_path)

In [ ]:
features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features]
y = df["Survived"]
num_cols = ["Age", "SibSp", "Parch", "Fare"]
cat_cols = ["Pclass", "Sex", "Embarked"]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000)),
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model.fit(X_train, y_train)
prob = model.predict_proba(X_valid)[:, 1]
print("Validation ROC AUC:", round(roc_auc_score(y_valid, prob), 3))

In [ ]:
for threshold in [0.30, 0.50, 0.70]:
    pred = (prob >= threshold).astype(int)
    print(
        threshold,
        "precision=", round(precision_score(y_valid, pred, zero_division=0), 3),
        "recall=", round(recall_score(y_valid, pred, zero_division=0), 3),
    )

thresholds = np.linspace(0.10, 0.90, 17)
precisions, recalls = [], []
for threshold in thresholds:
    pred = (prob >= threshold).astype(int)
    precisions.append(precision_score(y_valid, pred, zero_division=0))
    recalls.append(recall_score(y_valid, pred))

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(thresholds, precisions, marker="o", label="precision")
plt.plot(thresholds, recalls, marker="o", label="recall")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Threshold policy changes precision and recall")
plt.legend()
plt.tight_layout()
plt.show()

## Part B — K-nearest neighbors versus Gaussian naive Bayes
**Core activity.**


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "all_datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
train_path = PROJECT_ROOT / "all_datasets" / "breast_cancer_dataset" / "data.csv"
df = pd.read_csv(train_path)

In [ ]:
X = df.drop(columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore")
y = (df["diagnosis"] == "M").astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
# Select k only inside the training partition.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_values = [1, 3, 5, 7, 11, 21]
cv_scores = {}

for k in k_values:
    candidate = Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=k, weights="distance")),
    ])
    scores = cross_val_score(candidate, X_train, y_train, cv=cv, scoring="roc_auc")
    cv_scores[k] = scores.mean()
    print(f"k={k:2d} mean training-CV ROC AUC={scores.mean():.3f}")

best_k = max(cv_scores, key=cv_scores.get)
print("selected k:", best_k)

In [ ]:
models = {
    "KNN": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=best_k, weights="distance")),
    ]),
    "GaussianNB": Pipeline([
        ("scale", StandardScaler()),
        ("model", GaussianNB()),
    ]),
}

In [ ]:
# Validation compares the locked KNN choice with the alternative model family.
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    prob = model.predict_proba(X_valid)[:, 1]
    print(
        name,
        "validation accuracy=", round(accuracy_score(y_valid, pred), 3),
        "validation ROC AUC=", round(roc_auc_score(y_valid, prob), 3),
    )